In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

Extract the "multiqc_general_stats.txt" from the multiqc_data directory and place it in the reports dir in order to this script to work

In [ ]:
general_stats = pd.read_csv('../../reports/multiqc_general_stats.txt', sep='\t', index_col=0)
general_stats

In [ ]:
general_stats_GB = general_stats[general_stats.index.str.startswith("GB")]
general_stats_control = general_stats[~general_stats.index.str.startswith("GB")]

In [ ]:
def describe_stats(df):
	mean_std = df.agg(['mean', 'std']).round(2)
	median_iqr = df.agg([
		'median',
		lambda x: x.quantile(0.25),
		lambda x: x.quantile(0.75),
		lambda x: x.quantile(0.75) - x.quantile(0.25)
	]).round(2)
	median_iqr.index = ['median', 'Q1', 'Q3', 'IQR']
	print("Media y desviación típica:")
	print(mean_std)
	print("\nMediana y rango intercuartílico (IQR):")
	print(median_iqr)


In [ ]:
describe_stats(general_stats_GB)

In [ ]:
describe_stats(general_stats_control)

In [ ]:
describe_stats(general_stats)

Extract the "fastqc_sequence_length_distribution_plot.txt" from the multiqc_data directory and place it in the reports dir in order to this script to work

In [ ]:
fastqc_length = pd.read_csv('../reports/fastqc_sequence_length_distribution_plot.txt', sep='\t', index_col=0)
fastqc_length

In [ ]:
length_stats_GB = fastqc_length[fastqc_length.index.str.startswith("GB")]
length_stats_CV = fastqc_length.loc[fastqc_length.index.str.startswith("CV")]

In [ ]:
gb_medians = length_stats_GB['TOTAL'].str.replace(',', '.').astype(float)
cv_medians = length_stats_CV['TOTAL'].str.replace(',', '.').astype(float)


In [ ]:
from scipy.stats import mannwhitneyu

In [ ]:
df = pd.DataFrame({
    'valor': pd.concat([gb_medians, cv_medians], ignore_index=True),
    'grupo': ['GB'] * len(gb_medians) + ['CV'] * len(cv_medians)
})

In [ ]:
colors = ['mediumpurple', 'orange']
orden = ['GBM', 'CV']

In [ ]:
# Crear gráfico base
plt.figure(figsize=(7, 5))
sns.violinplot(data=df, x='grupo', y='valor', inner=None, cut=0, linewidth=0,
               palette=colores, order=orden)

sns.boxplot(data=df, x='grupo', y='valor', width=0.15, palette=colores, order=orden,
            boxprops={'zorder': 2}, showcaps=True, showfliers=False, whiskerprops={'linewidth': 1})

sns.stripplot(data=df, x='grupo', y='valor', size=4, color='k', alpha=0.5, order=orden)

# === Estadística ===
comparaciones = [('GBM', 'CV'), ('GBM', 'DC'), ('CV', 'DC')]
comparaciones = [('GBM', 'CV')]
posiciones_y = [df['valor'].max() + i*2 for i in range(len(comparaciones))]

for (grupo1, grupo2), y in zip(comparaciones, posiciones_y):
    datos1 = df[df['grupo'] == grupo1]['valor']
    datos2 = df[df['grupo'] == grupo2]['valor']
    stat, p = mannwhitneyu(datos1, datos2, alternative='two-sided')
    
    # Significancia como texto
    if p < 0.001:
        signif = '***'
    elif p < 0.01:
        signif = '**'
    elif p < 0.05:
        signif = '*'
    else:
        signif = 'ns'
    
    # Coordenadas
    x1 = orden.index(grupo1)
    x2 = orden.index(grupo2)
    
    # Dibujar línea de comparación
    plt.plot([x1, x1, x2, x2], [y-0.1, y, y, y-0.1], lw=1.2, color='grey')
    plt.text((x1 + x2) / 2, y - 1, signif, ha='center', va='bottom', fontsize=12)

# Personalización
sns.despine()
plt.xlabel('')
plt.ylabel('TOTAL')
plt.title('Longitud media de las reads por grupo')
plt.tight_layout()
plt.show()

In [ ]:
stat, p_value = mannwhitneyu(gb_medians.values, cv_medians.values, alternative='two-sided')
print(f"Mann-Whitney U test statistic: {stat:.2f}, p-value: {p_value:.10f}")